# SmokeyNet con el dataset original y etiquetas oficiales por tile

Este notebook entrena la arquitectura `ResNet34 + LSTM + SpatialViT` con los splits del articulo y `labels_stats_90overlap.pkl`. Las positivas sin estadisticas oficiales se excluyen del entrenamiento; las negativas conservan sus 45 tiles a cero.

Ejecuta primero `diagnostic`. Cambia a `full` solo cuando el smoke test y una epoca completa terminen correctamente.

In [ ]:
# 1. Configuracion y persistencia
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

RUN_MODE = 'diagnostic'  # 'diagnostic' o 'full'
REPO_URL = 'https://github.com/PedroHT8/SmokeyNet.git'
REPO_DIR = Path('/content/SmokeyNet')
RAW_ROOT = Path('/content/data/raw_images')
DRIVE_ROOT = Path('/content/drive/MyDrive/TFM_SmokeyNet_PaperDataset')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print('Modo:', RUN_MODE)
print('Checkpoints:', DRIVE_ROOT)

In [ ]:
# 2. Clonar el fork e instalar temporalmente el soporte de etiquetas precomputadas
import subprocess
import shutil

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

main_text = (REPO_DIR / 'src/main.py').read_text(encoding='utf-8')
if '--tile-label-stats-path' not in main_text:
    patch_text = 'diff --git a/src/dynamic_dataloader.py b/src/dynamic_dataloader.py\nindex 548625b..b13eab6 100644\n--- a/src/dynamic_dataloader.py\n+++ b/src/dynamic_dataloader.py\n@@ -37,6 +37,7 @@ class DynamicDataModule(pl.LightningDataModule):\n                  \n                  raw_data_path=None, \n                  labels_path=None, \n+                 tile_label_stats_path=None,\n                  metadata_path=\'./data/metadata.pkl\',\n                  optical_flow_path=None,\n                  \n@@ -85,6 +86,13 @@ class DynamicDataModule(pl.LightningDataModule):\n            \n         self.raw_data_path = raw_data_path\n         self.labels_path = labels_path\n+        self.tile_label_stats_path = tile_label_stats_path\n+        self.tile_label_stats = None\n+        if tile_label_stats_path is not None:\n+            with open(tile_label_stats_path, \'rb\') as stats_file:\n+                self.tile_label_stats = pickle.load(stats_file)\n+            if not isinstance(self.tile_label_stats, dict):\n+                raise TypeError(\'tile_label_stats_path must contain a dictionary keyed by image name.\')\n         self.metadata = pickle.load(open(metadata_path, \'rb\'))\n         self.optical_flow_path = optical_flow_path\n         \n@@ -116,6 +124,30 @@ class DynamicDataModule(pl.LightningDataModule):\n         self.blur_augment = blur_augment\n         self.color_augment = color_augment\n         self.brightness_contrast_augment = brightness_contrast_augment\n+\n+        if self.tile_label_stats is not None:\n+            if self.is_object_detection:\n+                raise ValueError(\'Precomputed tile labels are only supported by tile-classification models.\')\n+            if self.num_tile_samples > 0:\n+                raise ValueError(\'Precomputed tile labels do not support --num-tile-samples.\')\n+            paper_geometry = ((1392, 1856), 1040, (224, 224), 20)\n+            configured_geometry = (\n+                self.resize_dimensions,\n+                self.crop_height,\n+                self.tile_dimensions,\n+                self.tile_overlap,\n+            )\n+            if configured_geometry != paper_geometry:\n+                raise ValueError(\n+                    \'labels_stats_90overlap.pkl requires paper geometry: \'\n+                    \'resize=(1392, 1856), crop_height=1040, \'\n+                    \'tile_dimensions=(224, 224), tile_overlap=20.\'\n+                )\n+            if self.resize_crop_augment:\n+                raise ValueError(\n+                    \'Precomputed tile labels require fixed paper geometry. \'\n+                    \'Disable random crop jitter with --no-resize-crop-augment.\'\n+                )\n         \n         self.has_setup = False\n         \n@@ -187,6 +219,46 @@ class DynamicDataModule(pl.LightningDataModule):\n                 np.savetxt(log_dir+\'/train_images.txt\', self.train_split, fmt=\'%s\')\n                 np.savetxt(log_dir+\'/val_images.txt\', self.val_split, fmt=\'%s\')\n                 np.savetxt(log_dir+\'/test_images.txt\', self.test_split, fmt=\'%s\')\n+\n+        if self.tile_label_stats is not None:\n+            expected_num_tiles = util_fns.calculate_num_tiles(\n+                self.resize_dimensions,\n+                self.crop_height,\n+                self.tile_dimensions,\n+                self.tile_overlap)\n+            expected_num_tiles = expected_num_tiles[0] * expected_num_tiles[1]\n+            invalid_stats = [\n+                image_name for image_name, counts in self.tile_label_stats.items()\n+                if len(counts) != expected_num_tiles\n+            ]\n+            if invalid_stats:\n+                raise ValueError(\n+                    f\'Precomputed labels do not match the configured grid of \'\n+                    f\'{expected_num_tiles} tiles. Invalid entries: {invalid_stats[:5]}\'\n+                )\n+\n+            train_before = len(self.train_split)\n+            missing_positive_stats = [\n+                image_name for image_name in self.train_split\n+                if util_fns.get_ground_truth_label(image_name) == 1\n+                and image_name not in self.tile_label_stats\n+            ]\n+            if missing_positive_stats:\n+                missing_set = set(missing_positive_stats)\n+                self.train_split = [\n+                    image_name for image_name in self.train_split\n+                    if image_name not in missing_set\n+                ]\n+\n+            covered_positives = sum(\n+                util_fns.get_ground_truth_label(image_name) == 1\n+                for image_name in self.train_split\n+            )\n+            negatives = len(self.train_split) - covered_positives\n+            print(\'Precomputed tile-label summary:\')\n+            print(f\'- grid: {expected_num_tiles} tiles\')\n+            print(f\'- train images: {len(self.train_split)} ({covered_positives} positive, {negatives} negative)\')\n+            print(f\'- removed positive images without tile labels: {train_before - len(self.train_split)}\')\n         \n         self.has_setup = True\n         print("Setting Up Data Complete.")\n@@ -195,6 +267,7 @@ class DynamicDataModule(pl.LightningDataModule):\n     def train_dataloader(self):\n         train_dataset = DynamicDataloader(raw_data_path=self.raw_data_path,\n                                           labels_path=self.labels_path, \n+                                          tile_label_stats=self.tile_label_stats,\n                                           optical_flow_path=self.optical_flow_path,\n                                           split_name=\'train\',\n                                           \n@@ -231,6 +304,7 @@ class DynamicDataModule(pl.LightningDataModule):\n     def val_dataloader(self):\n         val_dataset = DynamicDataloader(raw_data_path=self.raw_data_path, \n                                           labels_path=self.labels_path, \n+                                          tile_label_stats=self.tile_label_stats,\n                                           optical_flow_path=self.optical_flow_path,\n                                           split_name=\'val\',\n                                         \n@@ -266,6 +340,7 @@ class DynamicDataModule(pl.LightningDataModule):\n     def test_dataloader(self):\n         test_dataset = DynamicDataloader(raw_data_path=self.raw_data_path, \n                                           labels_path=self.labels_path, \n+                                          tile_label_stats=self.tile_label_stats,\n                                           optical_flow_path=self.optical_flow_path,\n                                           split_name=\'test\',\n                                          \n@@ -308,6 +383,7 @@ class DynamicDataloader(Dataset):\n     def __init__(self, \n                  raw_data_path=None,\n                  labels_path=None, \n+                 tile_label_stats=None,\n                  optical_flow_path=None,\n                  split_name=\'train\',\n                  \n@@ -335,6 +411,7 @@ class DynamicDataloader(Dataset):\n         \n         self.raw_data_path = raw_data_path\n         self.labels_path = labels_path\n+        self.tile_label_stats = tile_label_stats\n         self.optical_flow_path = optical_flow_path\n         self.split_name = split_name\n         \n@@ -423,18 +500,37 @@ class DynamicDataloader(Dataset):\n         \n         ### Load Tile Labels ###\n         if self.is_maskrcnn or not self.is_object_detection:\n-            label_path = self.labels_path+\'/\'+image_name+\'.npy\'\n-            if Path(label_path).exists():\n+            has_precomputed_labels = (\n+                not self.is_object_detection\n+                and self.tile_label_stats is not None\n+                and image_name in self.tile_label_stats\n+            )\n+            if has_precomputed_labels:\n+                tile_counts = np.asarray(self.tile_label_stats[image_name])\n+                tiled_labels = (tile_counts > self.smoke_threshold).astype(float)\n+\n+                # Images and tiles are flattened row-major, so a horizontal flip\n+                # reverses the tile columns while preserving row order.\n+                if data_augmentations.should_flip:\n+                    tiled_labels = tiled_labels.reshape(\n+                        self.num_tiles_height, self.num_tiles_width\n+                    )[:, ::-1].reshape(-1).copy()\n+\n+                omit_mask = True\n+            else:\n+                label_path = None if self.labels_path is None else Path(self.labels_path) / f\'{image_name}.npy\'\n+\n+            if not has_precomputed_labels and label_path is not None and label_path.exists():\n                 # Repeat similar steps to images\n                 labels = np.load(label_path)\n \n                 labels = data_augmentations(labels, is_labels=True)\n-            else:\n+            elif not has_precomputed_labels:\n                 # Load all 0s\n                 # labels.shape = [height, width]\n                 labels = np.zeros((self.crop_height, self.resize_dimensions[1])).astype(float) \n \n-            if not self.is_object_detection:\n+            if not self.is_object_detection and not has_precomputed_labels:\n                 # Tile labels\n                 tiled_labels = util_fns.tile_labels(labels, self.num_tiles_height, self.num_tiles_width, self.resize_dimensions, self.tile_dimensions, self.tile_overlap)\n \n@@ -516,4 +612,4 @@ class DynamicDataloader(Dataset):\n         # Address ambiguous batch_size warning\n         image_name = image_name if self.split_name == \'test\' else 0\n         \n-        return image_name, x, tiled_labels, bbox_labels, ground_truth_label, omit_mask\n\\ No newline at end of file\n+        return image_name, x, tiled_labels, bbox_labels, ground_truth_label, omit_mask\ndiff --git a/src/main.py b/src/main.py\nindex fa0fb80..726502a 100644\n--- a/src/main.py\n+++ b/src/main.py\n@@ -62,6 +62,8 @@ parser.add_argument(\'--raw-data-path\', type=str, default=\'/root/raw_images\',\n                     help=\'Path to raw images.\')\n parser.add_argument(\'--labels-path\', type=str, default=\'/root/drive_clone_numpy\',\n                     help=\'Path to processed XML labels.\')\n+parser.add_argument(\'--tile-label-stats-path\', type=str, default=None,\n+                    help=\'Optional pickle containing per-image smoke-pixel counts for each tile. Positive training images without an entry are omitted.\')\n parser.add_argument(\'--metadata-path\', type=str, default=\'./data/metadata.pkl\',\n                     help=\'Path to metadata.pkl.\')\n parser.add_argument(\'--optical-flow-path\', type=str, default=None,\n@@ -215,6 +217,7 @@ def main(# Debug args\n         # Path args\n         raw_data_path=None, \n         labels_path=None, \n+        tile_label_stats_path=None,\n         metadata_path=None,\n         optical_flow_path=None,\n     \n@@ -308,6 +311,7 @@ def main(# Debug args\n         # Path args\n         raw_data_path=raw_data_path,\n         labels_path=labels_path,\n+        tile_label_stats_path=tile_label_stats_path,\n         metadata_path=metadata_path,\n         optical_flow_path=optical_flow_path,\n         \n@@ -500,6 +504,7 @@ if __name__ == \'__main__\':\n         # Path args - always used command line args for these\n         raw_data_path=args[\'raw_data_path\'],\n         labels_path=args[\'labels_path\'], \n+        tile_label_stats_path=args[\'tile_label_stats_path\'],\n         metadata_path=args[\'metadata_path\'],\n         optical_flow_path=args[\'optical_flow_path\'],\n         \n'
    patch_path = Path('/tmp/smokeynet_precomputed_labels.patch')
    patch_path.write_text(patch_text, encoding='utf-8')
    subprocess.run(
        ['git', 'apply', '--ignore-space-change', '--ignore-whitespace', str(patch_path)],
        cwd=REPO_DIR,
        check=True,
    )
    print('Parche de etiquetas precomputadas aplicado.')
else:
    print('El fork ya incluye etiquetas precomputadas; no se aplica parche.')

subprocess.run(['git', 'diff', '--check'], cwd=REPO_DIR, check=True)

In [ ]:
# 3. Dependencias compatibles con el fork adaptado
import sys
import subprocess

packages = [
    'pytorch-lightning>=2.2,<2.7',
    'torchmetrics>=1.3,<1.10',
    'transformers',
    'efficientnet-pytorch',
    'gtrxl-torch',
    'opencv-python-headless',
    'scikit-learn',
    'tensorboard',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch
import pytorch_lightning as pl
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Lightning:', pl.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')
if not torch.cuda.is_available():
    raise RuntimeError('Activa un runtime con GPU antes de continuar.')

In [ ]:
# 4. Descargar las 270 secuencias de los splits oficiales
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

SPLIT_DIR = REPO_DIR / 'data/final_split'
split_paths = {
    'train': SPLIT_DIR / 'train_images_final.txt',
    'val': SPLIT_DIR / 'val_images_final.txt',
    'test': SPLIT_DIR / 'test_images_final.txt',
}

def normalized_name(line):
    parts = line.strip().replace('\\', '/').split('/')
    return f'{parts[-2]}/{Path(parts[-1]).stem}'

required_images = {
    normalized_name(line)
    for split_path in split_paths.values()
    for line in split_path.read_text().splitlines()
    if line.strip()
}
fires = sorted({name.split('/')[0] for name in required_images})
RAW_ROOT.mkdir(parents=True, exist_ok=True)

def download_fire(fire):
    fire_dir = RAW_ROOT / fire
    if fire_dir.exists() and any(fire_dir.glob('*.jpg')):
        return fire, 'presente'
    archive = Path('/content') / f'{fire}.tgz'
    url = f'https://cdn.hpwren.ucsd.edu/HPWREN-FIgLib-Data/Tar/{fire}.tgz'
    subprocess.run(['wget', '-q', '--show-progress', '-O', str(archive), url], check=True)
    subprocess.run(['tar', '-xzf', str(archive), '-C', str(RAW_ROOT)], check=True)
    archive.unlink(missing_ok=True)
    return fire, 'descargado'

print(f'Descargando/verificando {len(fires)} incendios y {len(required_images)} imagenes...')
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(download_fire, fire) for fire in fires]
    for index, future in enumerate(as_completed(futures), 1):
        fire, status = future.result()
        if index % 10 == 0 or index == len(futures):
            print(f'{index}/{len(futures)}: {fire} ({status})')

missing_images = sorted(
    name for name in required_images if not (RAW_ROOT / f'{name}.jpg').exists()
)
print('Imagenes requeridas:', len(required_images))
print('Imagenes ausentes:', len(missing_images))
if missing_images:
    raise FileNotFoundError(f'Faltan imagenes del split. Primeras: {missing_images[:10]}')

In [ ]:
# 5. Preflight de etiquetas y balance
import pickle

STATS_PATH = REPO_DIR / 'data/label_stats/labels_stats_90overlap.pkl'
METADATA_PATH = REPO_DIR / 'data/metadata.pkl'
with STATS_PATH.open('rb') as handle:
    tile_stats = pickle.load(handle)

train_images = [
    normalized_name(line)
    for line in split_paths['train'].read_text().splitlines()
    if line.strip()
]
invalid = [name for name, counts in tile_stats.items() if len(counts) != 45]
positive = [name for name in train_images if '+' in name]
negative = [name for name in train_images if '+' not in name]
covered = [name for name in positive if name in tile_stats]
missing_positive = [name for name in positive if name not in tile_stats]
positive_tiles = sum(sum(value > 250 for value in tile_stats[name]) for name in covered)
negative_tiles = (len(covered) + len(negative)) * 45 - positive_tiles

summary = {
    'train_original': len(train_images),
    'positive_covered': len(covered),
    'negative_retained': len(negative),
    'positive_omitted': len(missing_positive),
    'train_effective': len(covered) + len(negative),
    'positive_tiles': positive_tiles,
    'negative_tiles': negative_tiles,
    'negative_positive_ratio': negative_tiles / positive_tiles,
}
print(summary)
assert not invalid
assert summary['train_effective'] == 10639
assert summary['positive_covered'] == 5013
assert 35 < summary['negative_positive_ratio'] < 38
print('Preflight correcto.')

In [ ]:
# 6. Smoke test del DataLoader antes de usar horas de GPU
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
from dynamic_dataloader import DynamicDataModule

dm = DynamicDataModule(
    omit_list=['omit_no_xml'],
    raw_data_path=str(RAW_ROOT),
    labels_path=None,
    tile_label_stats_path=str(STATS_PATH),
    metadata_path=str(METADATA_PATH),
    train_split_path=str(split_paths['train']),
    val_split_path=str(split_paths['val']),
    test_split_path=str(split_paths['test']),
    load_images_from_split=True,
    batch_size=1,
    num_workers=0,
    series_length=2,
    resize_dimensions=(1392, 1856),
    crop_height=1040,
    tile_dimensions=(224, 224),
    tile_overlap=20,
    smoke_threshold=250,
    resize_crop_augment=False,
    blur_augment=False,
    color_augment=False,
    brightness_contrast_augment=False,
)
dm.setup()
batch = next(iter(dm.train_dataloader()))
_, images, labels, _, image_gt, omit_mask = batch
print('images:', tuple(images.shape))
print('tile labels:', tuple(labels.shape), 'positive tiles:', int(labels.sum()))
print('image gt:', image_gt.tolist(), 'omit mask:', omit_mask.tolist())
assert images.shape[1:] == (45, 2, 3, 224, 224)
assert labels.shape[1] == 45
print('Smoke test correcto.')

In [ ]:
# 7. Argumentos de entrenamiento
import shlex

EXPERIMENT_NAME = f'smokeynet_paper_precomputed_{RUN_MODE}'
MAX_EPOCHS = 2 if RUN_MODE == 'diagnostic' else 25

base_args = [
    sys.executable, '-u', 'src/main.py',
    '--experiment-name', EXPERIMENT_NAME,
    '--experiment-description', 'SmokeyNet paper dataset with official precomputed tile statistics.',
    '--raw-data-path', str(RAW_ROOT),
    '--labels-path', '/content/unused_labels',
    '--tile-label-stats-path', str(STATS_PATH),
    '--metadata-path', str(METADATA_PATH),
    '--train-split-path', str(split_paths['train']),
    '--val-split-path', str(split_paths['val']),
    '--test-split-path', str(split_paths['test']),
    '--load-images-from-split',
    '--omit-list', 'omit_no_xml',
    '--error-as-eval-loss',
    '--model-type-list', 'RawToTile_ResNet', 'TileToTile_LSTM', 'TileToTileImage_SpatialViT',
    '--use-image-preds',
    '--series-length', '2',
    '--resize-height', '1392', '--resize-width', '1856', '--crop-height', '1040',
    '--tile-size', '224', '--tile-overlap', '20', '--smoke-threshold', '250',
    '--no-resize-crop-augment',
    '--batch-size', '1', '--accumulate-grad-batches', '32', '--num-workers', '0',
    '--tile-loss-type', 'bce', '--bce-pos-weight', '40', '--image-pos-weight', '5',
    '--optimizer-type', 'SGD', '--learning-rate', '0.001', '--optimizer-weight-decay', '0.001',
    '--min-epochs', '1', '--max-epochs', str(MAX_EPOCHS),
    '--no-early-stopping', '--gradient-clip-val', '1.0',
    '--tile-embedding-size', '1000', '--backbone-size', 'small',
]
if RUN_MODE == 'diagnostic':
    base_args.append('--no-stochastic-weight-avg')

print(' '.join(shlex.quote(str(arg)) for arg in base_args))

In [ ]:
# 8. Entrenar. La salida se muestra en directo y no queda oculta en capture_output.
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['PYTHONPATH'] = str(REPO_DIR / 'src')

subprocess.run(base_args, cwd=REPO_DIR, env=env, check=True)
print('Entrenamiento terminado correctamente.')

In [ ]:
# 9. Localizar y evaluar el mejor checkpoint explicitamente
checkpoint_dir = REPO_DIR / 'lightning_logs' / EXPERIMENT_NAME
best_candidates = sorted(
    [p for p in checkpoint_dir.glob('version_*/checkpoints/*.ckpt') if p.name != 'last.ckpt'],
    key=lambda p: p.stat().st_mtime,
)
last_candidates = sorted(
    checkpoint_dir.glob('version_*/checkpoints/last.ckpt'),
    key=lambda p: p.stat().st_mtime,
)
if not best_candidates:
    raise FileNotFoundError('No se encontro el best checkpoint.')

BEST_CKPT = best_candidates[-1]
LAST_CKPT = last_candidates[-1] if last_candidates else None
print('BEST_CKPT:', BEST_CKPT)
print('LAST_CKPT:', LAST_CKPT)

eval_args = base_args.copy()
eval_args[eval_args.index('--experiment-name') + 1] = EXPERIMENT_NAME + '_best_test'
eval_args += ['--checkpoint-path', str(BEST_CKPT), '--is-test-only']
subprocess.run(eval_args, cwd=REPO_DIR, env=env, check=True)
print('Evaluacion del best checkpoint terminada.')

In [ ]:
# 10. Guardar checkpoints y logs en Drive
import shutil
from datetime import datetime

destination = DRIVE_ROOT / f'{EXPERIMENT_NAME}_{datetime.now():%Y%m%d_%H%M%S}'
destination.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_CKPT, destination / 'best.ckpt')
if LAST_CKPT is not None:
    shutil.copy2(LAST_CKPT, destination / 'last.ckpt')

log_source = BEST_CKPT.parents[1]
for event_file in log_source.glob('events.out.tfevents.*'):
    shutil.copy2(event_file, destination / event_file.name)
print('Resultados guardados en:', destination)